# Instella Reasoning — GPU Experiment Suite (split-friendly, resumable, Drive-persisted)

Run **one model per session** to fit inside a Colab session (~3 h each). All sessions write
to the same Drive folder, so the pieces assemble automatically.

**Set `MODELS` at the top of Cell 1** to the model for this session:
- `"instruct"` — Instella-3B-Instruct  (do this first — most important)
- `"math"` — Instella-3B-Math
- `"base"` — Instella-3B
- `"olmo"` — AMD-OLMo-1B
- `""` (empty) — all four in one go (~12 h; only if you can keep a session that long)

Robust to interruptions: **resumable** (finished model/benchmarks skipped) and
**Drive-persisted** (survives a full VM recycle). If Colab disconnects, just re-run Cell 1.

Prereqs: Runtime → GPU · Secrets (🔑): `github` + `HF_TOKEN` (Notebook-access ON). First run
asks to mount Google Drive — accept.

In [ ]:
MODELS = "instruct"   # one of: instruct · math · base · olmo · "" (all). Re-run to resume.
LIMIT = 200           # 20 = fast sanity pass · 200 = real smoke run

# ---- persist results to Drive so they survive a full VM recycle ----
from google.colab import drive
drive.mount('/content/drive')
RUN_DIR = f"/content/drive/MyDrive/instella-gpu-suite-L{LIMIT}"   # shared across model sessions

# ---- setup (idempotent: clone or pull, install only if missing) ----
import os, subprocess, glob, json
import torch
assert torch.cuda.is_available(), "No GPU: Runtime > Change runtime type > GPU, then restart."
print("GPU:", torch.cuda.get_device_name(0))

from google.colab import userdata
os.environ["GITHUB_TOKEN"] = userdata.get("github")
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
os.environ["HF_HUB_DISABLE_XET"] = "1"
os.environ["SUITE_MODELS"] = MODELS   # empty string => all models

if not os.path.isdir("/content/Instella-Reasoning"):
    !git clone -b agent/cpu-experiment-suite https://$GITHUB_TOKEN@github.com/GIND123/Instella-Reasoning /content/Instella-Reasoning
%cd /content/Instella-Reasoning
!git pull

if subprocess.run(["which", "instella-reasoning"], capture_output=True).returncode != 0:
    !pip install -q -e ".[hf,retrieval,viz,stats]"
    !pip install -q "bitsandbytes>=0.46.1"

# ---- run the selected model(s): resumable, writes to Drive RUN_DIR ----
!SUITE_MODELS="{MODELS}" bash experiments/run_gpu_suite.sh {LIMIT} "{RUN_DIR}"

# ---- accuracy quick-look across everything done so far (all sessions) ----
print(f"\n=== accuracy per (model x benchmark), LIMIT={LIMIT} ===")
for f in sorted(glob.glob(f"{RUN_DIR}/scores/*__*.jsonl")):
    if f.endswith("__ALL.jsonl"):
        continue
    rows = [json.loads(l) for l in open(f) if l.strip()]
    if rows:
        acc = sum(r["correct"] for r in rows) / len(rows)
        print(f"{f.split('/')[-1]:50} {len(rows):4} items  acc={acc:.3f}")

In [ ]:
# Results already persist on Drive under MyDrive/instella-gpu-suite-L<LIMIT>/.
# Grab a local zip if you want one:
LIMIT = 200
!cd /content/drive/MyDrive && zip -qr /content/gpu_results.zip instella-gpu-suite-L{LIMIT} && echo zipped
from google.colab import files
files.download("/content/gpu_results.zip")